# 02 - Preprocesamiento de Datos

## 1. Objetivo y alcance

El objetivo de este notebook es preparar conceptualmente el conjunto de datos para la etapa de modelado de Machine Learning, definiendo las transformaciones necesarias para cada tipo de variable y estableciendo el conjunto final de características que será utilizado posteriormente.

A partir de los resultados obtenidos durante el análisis exploratorio de datos, se evaluarán las características de las variables numéricas y categóricas para determinar qué técnicas de preprocesamiento son apropiadas para su utilización en los modelos.

El notebook comprenderá las siguientes actividades:

- Identificar y clasificar las variables según su naturaleza.
- Evaluar las transformaciones necesarias para las variables numéricas.
- Definir el tratamiento de las variables categóricas.
- Probar las transformaciones seleccionadas para verificar su funcionamiento.
- Evaluar y definir el conjunto final de variables que pasarán a la etapa de modelado.

El preprocesamiento realizado en este notebook tiene un propósito experimental y de validación. Las transformaciones se aplicarán únicamente para comprobar su comportamiento y no representan todavía el pipeline definitivo utilizado para entrenar los modelos.

Por lo tanto, en esta etapa no se realizará:

- Train/Test Split.
- Entrenamiento de modelos.
- Evaluación del desempeño predictivo.
- Aplicación definitiva del pipeline de preprocesamiento.

La selección final de variables se realizará considerando los resultados del EDA, la naturaleza de cada variable y su posible utilidad para el problema de predicción de churn. Esta selección se realizará sobre las variables en su representación original, antes de aplicar las transformaciones de preprocesamiento.

El pipeline definitivo será construido y aplicado posteriormente, durante la etapa de modelado, después de realizar el Train/Test Split. Esto permitirá que cualquier transformación que requiera aprendizaje de parámetros se ajuste exclusivamente con los datos de entrenamiento y se aplique posteriormente al conjunto de prueba, evitando data leakage.

## 2. Importaciónn y Carga del Dataset

In [16]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Load the autoreload extension to automatically reload modules when they change
# Set autoreload mode to 2: reload all modules before executing user code
%load_ext autoreload
%autoreload 2 

# Import diagnostic functions
from src.features.diagnose_features import (
    build_numeric_diagnostics,
    build_categorical_diagnostics
)

from src.features.compare_transformatios import compare_transformations

# Load dataset
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 3. Definición de variables

Antes de definir las transformaciones de preprocesamiento, es necesario clasificar las variables de acuerdo con su naturaleza y representación dentro del dataset.

Esta clasificación permitirá determinar qué tipo de transformación es apropiada para cada variable. En este proyecto, las variables predictoras se dividirán en variables numéricas y categóricas.

### 3.1 Variables numéricas

Las variables numéricas representan cantidades en las que los valores tienen un significado cuantitativo y cuya magnitud y diferencia pueden ser interpretadas numéricamente.

En este proyecto se consideran variables numéricas:

- `tenure`
- `MonthlyCharges`
- `TotalCharges`

### 3.2 Variables categóricas

Las variables categóricas representan características cuyos valores corresponden a categorías o grupos.

En este proyecto se consideran variables categóricas:

- `gender`
- `SeniorCitizen`
- `Partner`
- `Dependents`
- `PhoneService`
- `MultipleLines`
- `InternetService`
- `OnlineSecurity`
- `OnlineBackup`
- `DeviceProtection`
- `TechSupport`
- `StreamingTV`
- `StreamingMovies`
- `Contract`
- `PaperlessBilling`
- `PaymentMethod`

Aunque `SeniorCitizen` está representada mediante los valores numéricos `0` y `1`, se considera una variable categórica binaria, ya que estos valores representan categorías y no cantidades. Por lo tanto, será procesada junto con las demás variables categóricas.

### 3.3 Variable objetivo

La variable objetivo del problema es `Churn`, que indica si el cliente abandonó o permaneció en el servicio.

Esta variable no será sometida al mismo proceso de transformación que las variables predictoras. Su tratamiento se definirá posteriormente como parte de la preparación del problema de clasificación binaria.

## 4. Preprocesamiento de variables numéricas

En esta sección se abordará el diagnóstico y evaluación de las variables numéricas, con el propósito de identificar sus características estadísticas y definir las transformaciones y tratamientos más adecuados para cada una. A partir del diagnóstico estadístico de cada variable, se analizarán aspectos como la presencia de valores nulos, la forma de la distribución, la existencia de valores extremos y su relación con la variable objetivo.

El diagnóstico permitirá evaluar diferentes características de cada variable, entre ellas:

- Valores faltantes.
- Asimetría de la distribución.
- Curtosis.
- Desviación estándar y rango de valores.
- Presencia de valores atípicos.
- Relación con la variable objetivo.
- Posibles transformaciones de la distribución.
- Necesidad de escalamiento.

A partir de estos resultados se definirán las técnicas de preprocesamiento para cada variable numérica. El objetivo no es asumir que todas las variables requieren el mismo tratamiento, sino identificar las características que justifican cada transformación y evaluar su comportamiento antes de construir el pipeline definitivo.

Posteriormente, las transformaciones seleccionadas serán implementadas y probadas de manera experimental sobre el conjunto de datos. Esta etapa permitirá verificar que las transformaciones funcionan correctamente y que producen una representación adecuada de las variables para su utilización posterior en los modelos.

### 4.1 Data Cleaning de `TotalCharges`

La variable `TotalCharges` se encuentra originalmente almacenada como texto debido a la presencia de 11 registros con valores vacíos. Durante el análisis exploratorio se identificó que estos 11 registros corresponden a clientes con `tenure = 0`. En estos casos, los clientes no presentan antigüedad dentro del servicio y, por consecuencia, no cuentan con cargos acumulados registrados.

Por lo tanto, los valores vacíos no representan datos faltantes que deban ser imputados mediante una medida estadística, sino una condición específica de los datos. En, por lo tanto, estos registros serán convertidos a un valor de 0, representando la ausencia de cargos acumulados.

La variable `TotalCharges` será convertida a un tipo de dato numérico, transformando los valores no convertibles en valores nulos. Posteriormente, los valores nulos resultantes serán reemplazados por 0.

In [18]:
# Copy the original DataFrame for cleaning
df_prepared = df.copy()

# Convert empty strings and non-numeric values to NaN
df_prepared['TotalCharges'] = pd.to_numeric(df_prepared['TotalCharges'], errors="coerce")

# Replace missing charges with 0 for customers with no tenure
df_prepared['TotalCharges'] = df_prepared['TotalCharges'].fillna(0)

# Validate TotalCharges cleaning and check for remaining missing values
print(df_prepared[['tenure', 'TotalCharges']][df_prepared['TotalCharges']==0])
print()
print(f"Missing values in 'TotalCharges': {df_prepared['TotalCharges'].isna().sum()}")


      tenure  TotalCharges
488        0           0.0
753        0           0.0
936        0           0.0
1082       0           0.0
1340       0           0.0
3331       0           0.0
3826       0           0.0
4380       0           0.0
5218       0           0.0
6670       0           0.0
6754       0           0.0

Missing values in 'TotalCharges': 0


### 4.2 Diagnóstico de variables numéricas

Después de realizar el data cleaning de `TotalCharges`, se ejecutó el diagnóstico de las variables numéricas para evaluar sus características estadísticas y determinar las transformaciones de preprocesamiento potencialmente necesarias. El diagnóstico considera la presencia de valores nulos, asimetría, curtosis, dispersión, rango de valores, valores atípicos según el Criterio de Tukey y asociación con la variable objetivo mediante la correlación de Spearman.

In [19]:
# Define numerical features to analyze
numeric_features = df_prepared[['tenure', 'MonthlyCharges', 'TotalCharges']]

# Convert text categories (Yes/No) to binary numbers (1/0)
df_prepared['Churn_num'] = df_prepared['Churn'].map({'Yes': 1, 'No': 0})

# Display diagnostics for numeric features (skewness, outliers, correlation, etc.)
print("\n ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================\n")
build_numeric_diagnostics(df_prepared, numeric_features, 'Churn_num')



 ======================== DIAGNOSTIC OF NUMERICAL FEATURES ========================



,feature,nulls_%,skew,kurtosis,std,min,max,outliers_%,corr_with_target,signal_suggestion,null_treatment_suggestion,transform_suggestion,binning_suggestion,scaling_suggestion
0,tenure,0.0,0.239540,-1.387372,24.559481,0.00,72.00,0.0,-0.367062,useful_signal,no_impute,no_transform,no_binning,standard
1,MonthlyCharges,0.0,-0.220524,-1.257260,30.090047,18.25,118.75,0.0,0.184743,useful_signal,no_impute,no_transform,no_binning,standard
2,TotalCharges,0.0,0.963235,-0.228580,2266.794470,0.00,8684.80,0.0,-0.229955,useful_signal,no_impute,no_transform,no_binning,minmax


1. `tenure`

La variable `tenure` no presenta valores nulos ni valores atípicos según el Criterio de Tukey. Su asimetría es baja (0.24), por lo que su distribución no presenta un sesgo importante. La curtosis negativa (-1.39) indica una distribución relativamente más plana que una distribución normal.

La correlación de Spearman con `Churn` es de -0.367, lo que indica una asociación negativa de magnitud moderada. Esto es consistente con los resultados obtenidos durante el EDA, donde se observó una menor proporción de churn conforme aumenta la antigüedad del cliente.

Debido a que no presenta problemas importantes de distribución, valores faltantes o valores atípicos, no se identifica la necesidad de aplicar una transformación sobre la variable. Sin embargo, debido a su escala numérica, se considera candidata para estandarización.


2. `MonthlyCharges`

La variable `MonthlyCharges` tampoco presenta valores nulos ni valores atípicos. Su asimetría es baja (-0.22), indicando una distribución aproximadamente simétrica, mientras que su curtosis negativa (-1.26) refleja una distribución más plana que una distribución normal.

La correlación de Spearman con `Churn` es de 0.185, lo que representa una asociación positiva de magnitud baja. Este resultado es consistente con el EDA, donde se identificó una diferencia estadísticamente significativa entre los cargos mensuales de clientes que abandonaron y aquellos que permanecieron, aunque con un tamaño de efecto pequeño.

No se observa una necesidad clara de transformar la distribución de la variable. Al igual que `tenure`, se considera conveniente evaluar su estandarización para los modelos sensibles a la escala de las variables.

3. `TotalCharges`

La variable `TotalCharges` no presenta valores nulos. La variable muestra una asimetría positiva moderada (0.96), indicando una concentración relativamente mayor de observaciones en valores bajos y una cola hacia valores altos.

A pesar de esta asimetría, no se identifican valores atípicos. Su desviación estándar es considerablemente mayor que la de las otras variables debido a la naturaleza acumulativa de los cargos.

La correlación de Spearman con `Churn` es de -0.230, indicando una asociación negativa de magnitud baja. Este resultado debe interpretarse junto con el análisis realizado durante el EDA, donde `TotalCharges` presentó una diferencia entre los grupos de churn y un tamaño de efecto moderado.

La función de diagnóstico sugiere `MinMaxScaler` y no identifica una transformación específica de la distribución como necesaria. Sin embargo, debido a la asimetría observada, se evaluará si una transformación logarítmica aporta alguna mejora antes de establecer el tratamiento definitivo. Esta decisión deberá considerar también la naturaleza acumulativa de la variable y su alta correlación con `tenure`.

En términos generales, las tres variables numéricas presentan una cantidad nula de valores faltantes y no muestran valores atípicos según el criterio de Tukey.

- `tenure` y `MonthlyCharges` presentan niveles bajos de asimetría, por lo que no se identifica una necesidad clara de transformación. `TotalCharges` presenta una mayor asimetría positiva, por lo que será la principal candidata para evaluar una transformación adicional.

- Las tres variables presentan asociación con `Churn`, aunque con diferente magnitud. `tenure` presenta la asociación negativa más elevada (-0.367), seguida de `TotalCharges` (-0.230) y `MonthlyCharges` (0.185).

### 4.3 Comparación de transformaciones en variables numéricas

Después del diagnóstico realizado en la sección anterior, se evaluarán diferentes transformaciones sobre las variables numéricas para determinar si alguna de ellas permite obtener una representación más adecuada para la etapa de modelado. La comparación se realizará entre la representación original de cada variable, una transformación logarítmica y una transformación de raíz cuadrada. Para cada alternativa se analizarán principalmente los cambios en la asimetría de la distribución, la presencia de valores atípicos y la asociación con la variable objetivo.

El objetivo de esta comparación no es buscar una distribución perfectamente normal, sino identificar si una transformación aporta una mejora relevante en la representación de la variable que justifique su incorporación al pipeline de preprocesamiento.

Las transformaciones evaluadas en esta sección tienen un propósito experimental. La decisión final sobre su aplicación se establecerá a partir de los resultados obtenidos y posteriormente serán incorporadas al pipeline de preprocesamiento durante la etapa de modelado.

In [20]:
compare_transformations(df_prepared, numeric_features, 'Churn_num' )


========== VARIABLE: tenure ==========

  transformation     mean      std  outliers_%    skew  corr_with_target
0       original  32.3711  24.5595         0.0  0.2395           -0.3522
1            log   3.0369   1.1555         0.0 -0.7784           -0.3733
2           sqrt   5.1146   2.4926         0.0 -0.2360           -0.3697

========== VARIABLE: MonthlyCharges ==========

  transformation     mean      std  outliers_%    skew  corr_with_target
0       original  64.7617  30.0900         0.0 -0.2205            0.1934
1            log   4.0433   0.5803         0.0 -0.7190            0.2101
2           sqrt   7.7812   2.0532         0.0 -0.4846            0.2034

========== VARIABLE: TotalCharges ==========

  transformation       mean        std  outliers_%    skew  corr_with_target
0       original  2279.7343  2266.7945      0.0000  0.9632           -0.1983
1            log     6.9325     1.5694      0.0016 -0.8245           -0.2340
2           sqrt    40.9371    24.5759      0.00

La comparación permite evaluar si las transformaciones logarítmica y de raíz cuadrada producen una mejora relevante respecto a la representación original de las variables numéricas. El análisis se centró principalmente en la asimetría de las distribuciones y, como referencia adicional, en el cambio de la asociación con la variable objetivo. La correlación se utiliza únicamente como indicador exploratorio y no como criterio único para seleccionar una transformación.

1. `tenure`

La representación original de `tenure` presenta una asimetría de 0.240, por lo que la distribución ya se encuentra relativamente cercana a la simetría. La transformación logarítmica modifica considerablemente la distribución y produce una asimetría negativa de -0.778, alejándola de la simetría observada en la representación original. La transformación de raíz cuadrada presenta un resultado similar al original, con una asimetría de -0.236.

En cuanto a la asociación con `Churn`, las tres representaciones producen valores similares: -0.352 en la versión original, -0.373 con logaritmo y -0.370 con raíz cuadrada. Por lo tanto, las transformaciones no generan una mejora relevante en este aspecto.

Debido a que la variable original ya presenta una distribución adecuada y las transformaciones no aportan una mejora significativa, se mantendrá `tenure` en su representación original.

2. `MonthlyCharges`

La variable `MonthlyCharges` presenta una asimetría de -0.221 en su representación original, indicando una distribución relativamente simétrica. La transformación logarítmica incrementa la asimetría negativa hasta -0.719, mientras que la raíz cuadrada la incrementa hasta -0.485. En ambos casos, la distribución se aleja de la representación original.

La asociación con `Churn` presenta cambios pequeños entre las alternativas: 0.193 en la representación original, 0.210 con logaritmo y 0.203 con raíz cuadrada. Aunque las transformaciones producen una ligera modificación en esta asociación, la diferencia no es suficiente para justificar un cambio de representación.

Por lo tanto, `MonthlyCharges` también se mantendrá en su representación original.

3. `TotalCharges`

La variable `TotalCharges` presenta el comportamiento más diferente de las tres variables. Su representación original muestra una asimetría positiva de 0.963, indicando una distribución con una cola hacia valores altos. La transformación logarítmica reduce inicialmente esta asimetría, pero produce una asimetría negativa de -0.825. Esto indica una sobrecorrección de la distribución.

La transformación de raíz cuadrada reduce la asimetría hasta 0.308, manteniendo el signo positivo pero produciendo una distribución considerablemente más equilibrada que la original. Además, no incrementa el porcentaje de valores identificados como atípicos por el criterio utilizado.

La asociación con `Churn` también presenta cambios moderados: -0.198 en la representación original, -0.234 con logaritmo y -0.223 con raíz cuadrada. Aunque la transformación logarítmica produce la mayor magnitud de asociación, también genera una sobrecorrección considerable de la distribución.

Por lo tanto, la transformación de raíz cuadrada representa una alternativa más equilibrada para `TotalCharges`, ya que mejora notablemente la asimetría sin producir la sobrecorrección observada con el logaritmo.

A partir de la comparación realizada, no se identifica una necesidad de transformar todas las variables numéricas.

El tratamiento candidato para cada variable será:

- `tenure`: Original
- `MonthlyCharges`: Original
- `TotalCharges`: Raíz cuadrada

Esta decisión busca mantener las transformaciones únicamente cuando aportan una mejora clara en la representación de la variable. En particular, `tenure` y `MonthlyCharges` presentan distribuciones suficientemente equilibradas en su forma original, mientras que `TotalCharges` sí muestra una mejora relevante al aplicar la raíz cuadrada.

La transformación seleccionada para `TotalCharges` se incorporará posteriormente al pipeline de preprocesamiento durante la etapa de modelado. De esta manera, la misma transformación podrá aplicarse de forma consistente al conjunto de entrenamiento, al conjunto de prueba y a nuevos datos.

## 5. Preprocesamiento de Variables Categóricas

## 6. Selección Final de Variables

## 7. Conclusiones y Próximos Pasos